In [1]:
import time
from pyspark.sql import SparkSession

# Creamos una sesión configurada explícitamente con 4 Hilos (Cores)
# master("local[4]") simula tener 4 CPUs disponibles.
spark = SparkSession.builder \
    .appName("Demo Particiones vs Cores") \
    .master("local[4]") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Cluster iniciado. Nivel de paralelismo por defecto: {sc.defaultParallelism}")
print("Tus 'Cajas Registradoras' (Slots) disponibles son: 4")

Cluster iniciado. Nivel de paralelismo por defecto: 4
Tus 'Cajas Registradoras' (Slots) disponibles son: 4


In [16]:
def tarea_pesada(iterator):
    # Simulamos que procesar esta partición toma 1 segundo
    time.sleep(1) 
    yield sum(1 for _ in iterator) # Devolvemos un conteo simple

def ejecutar_experimento(num_particiones):
    print(f"\n--- Iniciando prueba con {num_particiones} particiones ---")
    
    # Creamos un RDD con N particiones
    # El rango no importa mucho, lo importante es el numSlices
    rdd = sc.parallelize(range(1000), numSlices=num_particiones)

    print(rdd.getNumPartitions())
    
    inicio = time.time()
    # Acción forzada: count() para disparar el trabajo
    resultado = rdd.mapPartitions(tarea_pesada).collect()
    print(resultado)
    fin = time.time()
    
    tiempo_total = fin - inicio
    print(f"Tiempo total: {tiempo_total:.2f} segundos")
    return tiempo_total

In [17]:
# Caso: Tenemos menos trabajo que capacidad
t_a = ejecutar_experimento(num_particiones=2)
print("Análisis: Los 4 cores estaban listos, pero solo usaste 2. Desperdiciaste el 50% de capacidad.")


--- Iniciando prueba con 2 particiones ---
2
[500, 500]
Tiempo total: 1.11 segundos
Análisis: Los 4 cores estaban listos, pero solo usaste 2. Desperdiciaste el 50% de capacidad.


In [18]:
# Caso: Ajuste perfecto
t_b = ejecutar_experimento(num_particiones=4)
print("Análisis: Todos los cores trabajaron en paralelo. Máxima eficiencia.")


--- Iniciando prueba con 4 particiones ---
4
[250, 250, 250, 250]
Tiempo total: 1.15 segundos
Análisis: Todos los cores trabajaron en paralelo. Máxima eficiencia.


In [19]:
# Caso: El doble de trabajo que de capacidad
t_c = ejecutar_experimento(num_particiones=8)
print("Análisis: Se ejecutó en 2 Olas (Waves). 4 tareas primero, 4 tareas después.")


--- Iniciando prueba con 8 particiones ---
8
[125, 125, 125, 125, 125, 125, 125, 125]
Tiempo total: 2.24 segundos
Análisis: Se ejecutó en 2 Olas (Waves). 4 tareas primero, 4 tareas después.


In [6]:
# Caso: Fragmentación excesiva
t_d = ejecutar_experimento(num_particiones=40)
olas_teoricas = 40 / 4
print(f"Análisis: Tuviste que hacer {olas_teoricas} turnos (olas) para terminar.")


--- Iniciando prueba con 40 particiones ---
Tiempo total: 11.05 segundos
Análisis: Tuviste que hacer 10.0 turnos (olas) para terminar.
